# Service Checks und Hardcoded Tasks

Dieses Notebook ist das zentrale Kontrollpult fuer den aktuellen lokalen Experimentstand:

1. Docker-Service-Status pruefen
2. Services starten, falls noetig
3. BrowserGym-Service-Probes ausfuehren
4. hardcodierte Beispielaufgaben pro Site ausfuehren
5. Outputs als Tabellen anzeigen

`map` ist bewusst ausgeschlossen, weil der lokale Speicherbedarf zu gross ist.

In [12]:
from pathlib import Path
import json
import subprocess
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

OFFICIAL_REPO = PROJECT_ROOT / 'external' / 'webarena-verified'
SERVICE_PROBE_DIR = OFFICIAL_REPO / 'output' / 'service-probe'
HARDCODED_DIR = OFFICIAL_REPO / 'output' / 'hardcoded-tasks'

DEFAULT_SITES = ['shopping', 'shopping_admin', 'reddit', 'gitlab', 'wikipedia']

PROJECT_ROOT, OFFICIAL_REPO

(PosixPath('/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code'),
 PosixPath('/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified'))

In [13]:
def run_cmd(args, check=False):
    result = subprocess.run(args, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print('$', ' '.join(map(str, args)))
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with code {result.returncode}')
    return result

def read_json(path: Path):
    return json.loads(path.read_text())

def read_jsonl(path: Path):
    if not path.exists():
        return pd.DataFrame()
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    return pd.DataFrame(rows)

def show_json(path: Path):
    print(path)
    return read_json(path)

## 1. Services starten oder pruefen

Diese Zelle startet fehlende Services. Wenn bereits alles laeuft, meldet das Skript nur `OK`. Wikipedia wird explizit eingeschlossen.

In [14]:
RUN_START_SERVICES = False

if RUN_START_SERVICES:
    run_cmd([
        'python', 'scripts/start_enabled_services.py',
        '--sites', *DEFAULT_SITES,
        '--include-wikipedia',
    ], check=True)
else:
    print('Start uebersprungen. Setze RUN_START_SERVICES = True, wenn fehlende Services automatisch gestartet werden sollen.')

Start uebersprungen. Setze RUN_START_SERVICES = True, wenn fehlende Services automatisch gestartet werden sollen.


## 2. Service-Probe ausfuehren

Der Service-Probe prueft Docker-Status, rendert pro Site eine Beispielaufgabe oder direkte Startseite und oeffnet sie mit BrowserGym. Das ist noch keine Task-Loesung.

In [15]:
RUN_SERVICE_PROBE = True

if RUN_SERVICE_PROBE:
    run_cmd([
        'python', 'scripts/run_services_probe.py',
        '--sites', *DEFAULT_SITES,
    ], check=False)
else:
    print('Service-Probe uebersprungen.')

$ python scripts/run_services_probe.py --sites shopping shopping_admin reddit gitlab wikipedia

Service status
- OK: shopping container=webarena_verified_shopping status=running
- OK: shopping_admin container=webarena_verified_shopping_admin status=running
- OK: reddit container=webarena_verified_reddit status=running
- OK: gitlab container=wa-demo-gitlab status=running
- OK: wikipedia container=webarena_verified_wikipedia status=running
[TASK] shopping: task_id=118 type=NAVIGATE intent=I have a jaw bruxism problem, go to the product page for something that could alleviate the problem.
[OPEN] shopping: start_url=http://localhost:7770
[TASK] shopping_admin: task_id=157 type=NAVIGATE intent=View the details of all customers
[OPEN] shopping_admin: start_url=http://localhost:7780/admin
[TASK] reddit: task_id=27 type=RETRIEVE intent=In the personal finances forum, get the username and post title of the most recent post, and count the number of comments on that post that are not from the aut

## 3. Service-Probe Outputs

In [16]:
service_summary_path = SERVICE_PROBE_DIR / 'summary.json'
service_status_path = SERVICE_PROBE_DIR / 'service_status.json'
service_log_path = SERVICE_PROBE_DIR / 'probe_log.jsonl'

service_summary = read_json(service_summary_path)
pd.DataFrame(service_summary['results'])

,site,status,task_id,start_url,final_url,page_title,output_dir,task_intent,task_type,error
0,shopping,success,118.0,http://localhost:7770,http://localhost:7770/,One Stop Market,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,"I have a jaw bruxism problem, go to the produc...",NAVIGATE,None
1,shopping_admin,success,157.0,http://localhost:7780/admin,http://localhost:7780/admin,Magento Admin,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,View the details of all customers,NAVIGATE,None
2,reddit,success,27.0,http://localhost:9999,http://localhost:9999/,Postmill,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,"In the personal finances forum, get the userna...",RETRIEVE,None
3,gitlab,success,44.0,http://localhost:8012,http://localhost:8012/users/sign_in,Sign in · GitLab,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,Open my todos page,NAVIGATE,None
4,wikipedia,success,NaN,http://localhost:8888,http://localhost:8888/wikipedia_en_all_maxi_20...,User:The other Kiwix guy/Landing,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,Open the local wikipedia service start page.,SERVICE_PROBE,None


In [17]:
pd.DataFrame(read_json(service_status_path))

,site,container_name,running,docker_status,start_command
0,shopping,webarena_verified_shopping,True,running,"[uv, run, webarena-verified, env, start, --sit..."
1,shopping_admin,webarena_verified_shopping_admin,True,running,"[uv, run, webarena-verified, env, start, --sit..."
2,reddit,webarena_verified_reddit,True,running,"[uv, run, webarena-verified, env, start, --sit..."
3,gitlab,wa-demo-gitlab,True,running,"[uv, run, invoke, -r, examples, gitlab-start]"
4,wikipedia,webarena_verified_wikipedia,True,running,"[uv, run, webarena-verified, env, start, --sit..."


In [18]:
read_jsonl(service_log_path)

,event,site,task_type,config_path,task_id,intent,task_path,start_urls,status,start_url,final_url,page_title,output_dir,task_intent,error,reason
0,site_probe_started,shopping,NAVIGATE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,site_config_written,shopping,NaN,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,candidate_task_selected,shopping,NAVIGATE,NaN,118.0,"I have a jaw bruxism problem, go to the produc...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,agent_input_rendered,shopping,NaN,NaN,118.0,NaN,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,[http://localhost:7770],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,site_probe_finished,shopping,NAVIGATE,NaN,118.0,NaN,NaN,NaN,success,http://localhost:7770,http://localhost:7770/,One Stop Market,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,"I have a jaw bruxism problem, go to the produc...",NaN,NaN
5,site_probe_started,shopping_admin,NAVIGATE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,site_config_written,shopping_admin,NaN,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,candidate_task_selected,shopping_admin,NAVIGATE,NaN,157.0,View the details of all customers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,agent_input_rendered,shopping_admin,NaN,NaN,157.0,NaN,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,[http://localhost:7780/admin],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,site_probe_finished,shopping_admin,NAVIGATE,NaN,157.0,NaN,NaN,NaN,success,http://localhost:7780/admin,http://localhost:7780/admin,Magento Admin,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,View the details of all customers,NaN,NaN


## 4. Hardcoded Tasks ausfuehren

Diese Aufgaben fuehren pro Site eine deterministische Aktion aus. Fuer alle Aufgaben mit echter Task-ID wird die offizielle WebArena-Verified-Evaluation ausgefuehrt. GitLab Task 44 ist aktuell voll geloest; die anderen Sites pruefen zunaechst Evaluator-Contract, Navigation/Login/HAR/Artefaktstruktur und bekommen erwartbar noch keinen Erfolgsscore.

In [19]:
RUN_HARDCODED_TASKS = True

if RUN_HARDCODED_TASKS:
    run_cmd([
        'python', 'scripts/run_hardcoded_tasks.py',
        '--sites', *DEFAULT_SITES,
    ], check=False)
else:
    print('Hardcoded Tasks uebersprungen.')

$ python scripts/run_hardcoded_tasks.py --sites shopping shopping_admin reddit gitlab wikipedia

Service status
- OK: shopping container=webarena_verified_shopping status=running
- OK: shopping_admin container=webarena_verified_shopping_admin status=running
- OK: reddit container=webarena_verified_reddit status=running
- OK: gitlab container=wa-demo-gitlab status=running
- OK: wikipedia container=webarena_verified_wikipedia status=running
[RUN] shopping: task=118 type=NAVIGATE
[RUN] shopping_admin: task=157 type=NAVIGATE
[RUN] reddit: task=27 type=RETRIEVE
[RUN] gitlab: task=44 type=NAVIGATE
[RUN] wikipedia: task=direct type=SERVICE_PROBE

Summary: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/output/hardcoded-tasks/summary.json
- SUCCESS: shopping task=118 final_url=http://localhost:7770/dentemp-ora-guard-custom-fit-dental-guard-bruxism-night-guard-for-teeth-grinding-two-pack-mouth-guard-for-clenching-teeth-at-night-mouth-guard-for-sleep

## 5. Hardcoded Task Outputs

In [20]:
hardcoded_summary_path = HARDCODED_DIR / 'summary.json'
hardcoded_log_path = HARDCODED_DIR / 'hardcoded_log.jsonl'

hardcoded_summary = read_json(hardcoded_summary_path)
pd.DataFrame(hardcoded_summary['results'])

,site,status,task_id,task_type,intent,start_url,target_url,final_url,page_title,success,output_dir,error,official_score,total_runtime_ms,browser_runtime_ms,official_eval_runtime_ms,total_tokens
0,shopping,success,118.0,NAVIGATE,"I have a jaw bruxism problem, go to the produc...",http://localhost:7770,http://localhost:7770/dentemp-ora-guard-custom...,http://localhost:7770/dentemp-ora-guard-custom...,Dentemp Ora-GUARD Custom Fit Dental Guard - Br...,True,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,None,1.0,5701,5002,697.0,None
1,shopping_admin,success,157.0,NAVIGATE,View the details of all customers,http://localhost:7780/admin,http://localhost:7780/admin/customer/index/,http://localhost:7780/admin/customer/index/,Customers / Customers / Magento Admin,True,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,None,1.0,9248,8510,736.0,None
2,reddit,success,27.0,RETRIEVE,"In the personal finances forum, get the userna...",http://localhost:9999,http://localhost:9999/,http://localhost:9999/,Postmill,True,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,None,1.0,4771,4125,645.0,None
3,gitlab,success,44.0,NAVIGATE,Open my todos page,http://localhost:8012,http://localhost:8012/dashboard/todos,http://localhost:8012/dashboard/todos,To-Do List · GitLab,True,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,None,1.0,9013,8223,788.0,None
4,wikipedia,success,NaN,SERVICE_PROBE,Open the local Wikipedia landing page.,http://localhost:8888,NaN,http://localhost:8888/wikipedia_en_all_maxi_20...,User:The other Kiwix guy/Landing,True,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,None,NaN,2017,2017,NaN,None


In [21]:
read_jsonl(hardcoded_log_path)

,site,status,task_id,task_type,intent,start_url,target_url,final_url,page_title,success,output_dir,error,official_score,total_runtime_ms,browser_runtime_ms,official_eval_runtime_ms,total_tokens
0,shopping,success,118.0,NAVIGATE,"I have a jaw bruxism problem, go to the produc...",http://localhost:7770,http://localhost:7770/dentemp-ora-guard-custom...,http://localhost:7770/dentemp-ora-guard-custom...,Dentemp Ora-GUARD Custom Fit Dental Guard - Br...,True,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,None,1.0,5701,5002,697.0,None
1,shopping_admin,success,157.0,NAVIGATE,View the details of all customers,http://localhost:7780/admin,http://localhost:7780/admin/customer/index/,http://localhost:7780/admin/customer/index/,Customers / Customers / Magento Admin,True,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,None,1.0,9248,8510,736.0,None
2,reddit,success,27.0,RETRIEVE,"In the personal finances forum, get the userna...",http://localhost:9999,http://localhost:9999/,http://localhost:9999/,Postmill,True,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,None,1.0,4771,4125,645.0,None
3,gitlab,success,44.0,NAVIGATE,Open my todos page,http://localhost:8012,http://localhost:8012/dashboard/todos,http://localhost:8012/dashboard/todos,To-Do List · GitLab,True,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,None,1.0,9013,8223,788.0,None
4,wikipedia,success,NaN,SERVICE_PROBE,Open the local Wikipedia landing page.,http://localhost:8888,NaN,http://localhost:8888/wikipedia_en_all_maxi_20...,User:The other Kiwix guy/Landing,True,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,None,NaN,2017,2017,NaN,None


## 6. Artefaktpfade

Hier werden die wichtigsten Dateien pro hardcoded Task angezeigt.

In [22]:
artifact_rows = []
for metadata_path in HARDCODED_DIR.glob('*/*/hardcoded_metadata.json'):
    data = read_json(metadata_path)
    artifact_rows.append({
        'site': data['site'],
        'task_id': data['task_id'],
        'metadata': str(metadata_path),
        'network_har': data.get('network_har'),
        'trace': data.get('trace'),
        'agent_response': data.get('agent_response'),
        'eval_result': data.get('eval_result'),
    })

pd.DataFrame(artifact_rows).sort_values(['site', 'task_id'], na_position='last')

,site,task_id,metadata,network_har,trace,agent_response,eval_result
2,gitlab,44.0,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...
4,reddit,27.0,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...
0,shopping,118.0,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...
3,shopping_admin,157.0,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...
1,wikipedia,NaN,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,/Users/niclascramer/Privat/Uni/Uni-Reutlingen/...,NaN


## 7. Naechste Interpretation

- Wenn `service-probe` erfolgreich ist, laufen die lokalen Services und BrowserGym kann sie oeffnen.
- Wenn `hardcoded-tasks` erfolgreich ist, funktionieren pro Site deterministische Navigation, HAR und Artefaktstruktur.
- Die Spalte `official_score` zeigt, ob die konkrete Benchmark-Aufgabe bereits geloest wurde.
- GitLab Task 44 ist aktuell voll geloest; Shopping, Shopping-Admin und Reddit erzeugen valide Evaluationsartefakte, sind aber fachlich noch nicht geloest.
- Die naechste Entwicklungsstufe ist, diese hardcodierten Aufgaben in Planner/Executor/Evaluator/Controller-Laeufe zu ueberfuehren.